In [14]:
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv())

True

In [15]:
from langchain_core.messages import AIMessage
from langchain_core.tools import tool

from langgraph.prebuilt import ToolNode

In [16]:
from risk.assessment import build_assessment_chain

chain = build_assessment_chain()

@tool
def risk_assessment(location: str, activity: str):
    """Create a risk assessment for a given location and activity."""
    return chain.invoke({"location": location, "activity": activity})


@tool
def search(search: str):
    """Search for information."""
    return "I don't know the answer."

In [17]:
tools = [search, risk_assessment]
tool_node = ToolNode(tools)

In [18]:
from langchain_openai import ChatOpenAI

model = ChatOpenAI(model="gpt-4o-mini", temperature=0.5)

In [19]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a helpful assistant. Help the user create a risk assessment.",
        ),
        MessagesPlaceholder("history"),
        ("human", "{question}"),
    ]
)

In [20]:
app = prompt | model

In [21]:
def print_stream(stream):
    for s in stream:
        message = s["messages"][-1]
        if isinstance(message, tuple):
            print(message)
        else:
            message.pretty_print()

In [22]:
from langgraph.prebuilt import create_react_agent

graph = create_react_agent(model, tools=tools)

In [23]:
messages = [
    ("human", "Can you help me with a risk assessment.?"),
    (
        "ai",
        "'Sure! I can help you with a risk assessment. Please provide me with the following details:\n\n1. The location where you want the risk assessment to be conducted.\n2. The specific activity for which you need the risk assessment. \n\nOnce I have that information, I can proceed with the assessment.",
    ),
    ("human", "The location is a scout hut."),
    (
        "ai",
        "Great! Now, could you please specify the activity for which you need the risk assessment at the scout hut?",
    ),
    ("human", "The activity is pumpkin carving."),
]
messages = [
    ("human", "What is a wood badge?")
]
inputs = {"messages": messages}

In [24]:
# inputs = {"messages": [("human", "What is the weather like?")]}
# TODO the tool could return a question or a statement requiring clarification
print_stream(graph.stream(inputs, stream_mode="values"))

================================ Human Message =================================

What is a wood badge?
================================== Ai Message ==================================
Tool Calls:
  search (call_IRW20sLOcm839X8XgsFv9Abe)
 Call ID: call_IRW20sLOcm839X8XgsFv9Abe
  Args:
    search: wood badge
================================= Tool Message =================================
Name: search

I don't know the answer.
================================== Ai Message ==================================

A Wood Badge is a prestigious training award in the Boy Scouts of America (BSA) and other scouting organizations around the world. It is designed for adult leaders and volunteers involved in scouting, focusing on leadership skills, team building, and personal development. 

The Wood Badge program includes both practical and theoretical components, often culminating in a weekend course where participants engage in various scouting activities, leadership exercises, and discussions. Upon